## Install Python dependencies

In [1]:
!pip install --upgrade pip
# Install all requirements needed to train this model and track it in MLFlow.
!pip install pip -qU
!pip install mlflow<=2.8
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
/usr/bin/sh: line 1: =2.8: No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 153.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxruntime] [onnxruntime]


Import the dependencies for the model training code:

In [2]:
# Import the dependencies we need to run the code.
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import mlflow
import torch
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pickle
import torch.nn as nn
import torch.optim as optim
import random
from torch.utils.data import Dataset, DataLoader
import re
import os
import torch.onnx
import onnx
import onnxruntime as ort
#import mlflow.pyfunc
import time
from datetime import datetime, timezone

In [3]:
# Get the environment variable for how to reach MLFlow.
# The general format is http://<route-to-mlflow>:<port> where you replace <route-to-mlflow> and <port> with specifics for where your MLFlow instance is set up and/or exposed.

MLFLOW_ROUTE = os.getenv("MLFLOW_ROUTE")

## Load the CSV data

In [4]:
# csv_file = '../NERC-FixV2W/preprocess/'
csv_file = 'NERC-FixV2W/preprocess/'
# csv_file = 'saved'

In [5]:
df_cpe = pd.read_csv(csv_file + "csv_file/2021aug/cpe_ignore_version.csv", usecols=['part','vendor',
                'product', 'target_sw','target_hw','name','size']).sort_values(
                'size', ascending=False)
df_cwe = pd.read_csv(csv_file + "csv_file/2021aug/cwe.csv")
df_cwe.fillna('*', inplace=True)

df_cwe_cate = pd.read_csv(csv_file + "csv_file/2021aug/cwe_category.csv")
df_cwe_view = pd.read_csv(csv_file + "csv_file/2021aug/cwe_view.csv")
df_cwe_cate.fillna('*', inplace=True)
df_cwe_view.fillna('*', inplace=True)

In [6]:
SAVED_DIR = "saved"
def parse_folder_datetime(folder_name: str) -> datetime | None:
    try:
        iso_like = folder_name.replace("T", " ").replace("Z", "")
        date_part, time_part = iso_like.split(" ")

        h, m, s, ms = time_part.split("-")
        iso_string = f"{date_part}T{h}:{m}:{s}.{ms}Z"

        return datetime.strptime(
            iso_string, "%Y-%m-%dT%H:%M:%S.%fZ"
        ).replace(tzinfo=timezone.utc)

    except Exception:
        return None


def find_latest_folder() -> str:
    candidates = []

    for name in os.listdir(SAVED_DIR):
        path = os.path.join(SAVED_DIR, name)

        if not os.path.isdir(path):
            continue

        if not name.startswith("2"):
            continue

        dt = parse_folder_datetime(name)
        if dt:
            candidates.append((dt, name))

    return max(candidates, key=lambda x: x[0])[1]

end_date_folder = find_latest_folder()

output_dir = f"saved/{end_date_folder}/cve_reformatted.csv"

df_cve = pd.read_csv(output_dir)
df_cve = df_cve.fillna('*')


In [7]:
df_cve

,ID,last_modified,description,MatchingCWE,keywords,MatchingCPE
0,CVE-1999-0095,2025-04-03T01:03:51.193,"The debug command in Sendmail is enabled, allo...",NVD-CWE-Other,"['sendmail', 'debug', 'attackers']",cpe:2.3:a:eric_allman:sendmail:5.58:*:*:*:*:*:*:*
1,CVE-1999-0082,2025-04-03T01:03:51.193,CWD ~root command in ftpd allows root access.,NVD-CWE-Other,"['ftpd', 'root', 'cwd']",cpe:2.3:a:ftp:ftp:*:*:*:*:*:*:*:*;cpe:2.3:a:ft...
2,CVE-1999-1471,2025-04-03T01:03:51.193,Buffer overflow in passwd in BSD based operati...,NVD-CWE-Other,"['passwd', 'privileges', 'bsd']",cpe:2.3:o:bsd:bsd:4.2:*:*:*:*:*:*:*;cpe:2.3:o:...
3,CVE-1999-1122,2025-04-03T01:03:51.193,Vulnerability in restore in SunOS 4.0.3 and ea...,NVD-CWE-Other,"['privileges', 'vulnerability', 'restore']",cpe:2.3:o:sun:sunos:*:*:*:*:*:*:*:*;cpe:2.3:o:...
4,CVE-1999-1467,2025-04-03T01:03:51.193,Vulnerability in rcp on SunOS 4.0.x allows rem...,NVD-CWE-Other,"['rcp', 'sunos', 'vulnerability']",cpe:2.3:o:sun:sunos:4.0:*:*:*:*:*:*:*;cpe:2.3:...
...,...,...,...,...,...,...
327222,CVE-2026-33066,2026-03-23T15:31:10.727,*,*,[],cpe:2.3:a:b3log:siyuan:
327223,CVE-2026-33067,2026-03-23T15:31:25.547,*,*,[],cpe:2.3:a:b3log:siyuan:
327224,CVE-2026-33069,2026-03-23T15:32:13.853,*,*,[],cpe:2.3:a:pjsip:pjsip:
327225,CVE-2026-33070,2026-03-23T15:33:39.733,*,*,[],cpe:2.3:a:filerise:filerise:


In [8]:
# df_cve = pd.read_csv(csv_file + "csv_file/2021aug/cve/cve-2002.csv")
# df_cve = df_cve.astype('object').fillna('*')
# df_list = [df_cve]
# for i in range(2003, 2022):
#     filename = f'{csv_file}csv_file/2021aug/cve/cve-{i}.csv'
#     df_temp = pd.read_csv(filename)
#     df_temp = df_temp.astype('object').fillna('*')
#     df_list.append(df_temp)
# df_cve = pd.concat(df_list, ignore_index=True)

In [9]:

cwe_list = []

kg_cwe = []

for i, r in df_cwe.iterrows():
    cweid = r['ID']

    cwe_list.append(cweid)

    if (r['Related_Weakness'] != '*'):
        relatedlist = r['Related_Weakness'].split(';')
        for re in relatedlist:
            relation, other = re.split(':')
            other = 'CWE-' + other
            kg_cwe.append(([cweid, relation, other]))

    if (r['Language'] != '*'):
        langlist = r['Language'].split(';')
        for l in langlist:
            if (l != 'Language-Independent'):
                kg_cwe.append(([cweid, 'Language', l]))

    if (r['Technology'] != '*'):
        techlist = r['Technology'].split(';')
        for t in techlist:
            if (t != 'Technology-Independent'):
                kg_cwe.append(([cweid, 'Technology', t]))

    if (r['Likelihood_Of_Exploit'] != '*'):
        kg_cwe.append(([cweid, 'LikelihoodOfExploit', \
                        r['Likelihood_Of_Exploit']]))

    if (r['Consequence'] != '*'):
        conseqlist = r['Consequence'].split(';')
        for c in conseqlist:
            if (c != 'Other'):
                kg_cwe.append(([cweid, 'Consequence', c]))

kg_cate = []
kg_view = []

for i, r in df_cwe_cate.iterrows():
    cid = r['ID']

    if (r['Has_Member'] != '*'):
        memlist = r['Has_Member'].split(';')
        for m in memlist:
            kg_cate.append(([cid, 'Has_Member', m]))

for i, r in df_cwe_view.iterrows():
    vid = r['ID']

    if (r['Has_Member'] != '*'):
        memlist = r['Has_Member'].split(';')
        for m in memlist:
            kg_view.append(([vid, 'Has_Member', m]))

cpe_candidate_list = df_cpe['name'].values.tolist()
cve_candidate_list = []
connected_cvelist = []

kg_cve = []
kg_cpe2cve = []
kg_cve2cwe = []

connected_cpelist = []

for i, r in tqdm(df_cve.iterrows(), total = len(df_cve)):
    cveid = r['ID']
    cve_candidate_list.append(cveid)
    is_connected = 0

    if (r['MatchingCPE'] != '*'):
        cpes_related = r['MatchingCPE'].split(';')
        plist = []
        for p in cpes_related:

            p = p.replace('\,', '\.')
            p = p.replace('\:', '\;')
            p = p.replace('"', "'")

            p_detail = p.split(':')
            pp = ':'.join(['cpe']+p_detail[2:5]+p_detail[10:12])
            if (pp not in plist)  and (pp in cpe_candidate_list):
                matchcpe = [pp, 'MatchingCVE', cveid]
                kg_cpe2cve.append((matchcpe))
                is_connected += 1

            if pp not in connected_cpelist and (pp in cpe_candidate_list):
                connected_cpelist.append(pp)

            plist.append(pp)

    if (r['MatchingCWE'] != '*' and r['MatchingCWE'] != 'NVD-CWE-Other' and r['MatchingCWE'] != 'NVD-CWE-noinfo'):
        cwelist = r['MatchingCWE'].split(';')
        for w in cwelist:
            if (w != 'NVD-CWE-noinfo') and (w in cwe_list or w in df_cwe_cate['ID'].values.tolist() or w in df_cwe_view['ID'].values.tolist()):
                if w=='CWE-17':
                    print(cveid, w)
                matchcwe = [cveid, 'MatchingCWE', w]
                kg_cve2cwe.append((matchcwe))
                is_connected += 1

    if is_connected:
        connected_cvelist.append(cveid)
        is_connected = 0

kg_cve = kg_cpe2cve + kg_cve2cwe

  0%|          | 0/327227 [00:00<?, ?it/s]

In [10]:
with open(f'./saved/{end_date_folder}/cpelist_connected.txt', 'w') as f:
    for items in connected_cpelist:
        f.write('%s\n' %items)
    print("File written successfully")
f.close()

File written successfully


In [11]:
with open(f'./saved/{end_date_folder}/cvelist_connected.txt', 'w') as f:
    for items in connected_cvelist:
        f.write('%s\n' %items)
    print("File written successfully")
f.close()

File written successfully


In [12]:
f = open(f'./saved/{end_date_folder}/cpelist_connected.txt', 'r')
connected_cpelist = f.read().splitlines()
f.close()

In [13]:
f = open(f'./saved/{end_date_folder}/cvelist_connected.txt', 'r')
connected_cvelist = f.read().splitlines()
f.close()

In [14]:
cpe2cve_df = pd.DataFrame(kg_cpe2cve, columns=["subject", "predicate", "object"])
cpe2cve_df.to_csv(f'./saved/{end_date_folder}/cpe2cve-aug2021.csv', index=False)

cve2cwe_df = pd.DataFrame(kg_cve2cwe, columns=["subject", "predicate", "object"])
cve2cwe_df.to_csv(f'./saved/{end_date_folder}/cve2cwe-aug2021.csv', index=False)

In [15]:
# Load data
kg_cpe2cve_df = pd.read_csv(f"./saved/{end_date_folder}/cpe2cve-aug2021.csv")
kg_cve2cwe_df = pd.read_csv(f"./saved/{end_date_folder}/cve2cwe-aug2021.csv")

# Convert to list of triples using .values.tolist()
kg_cpe2cve = kg_cpe2cve_df[['subject', 'predicate', 'object']].values.tolist()
kg_cve2cwe = kg_cve2cwe_df[['subject', 'predicate', 'object']].values.tolist()

# Combine the two
kg_cve = kg_cpe2cve + kg_cve2cwe

In [16]:
kg_cpe = []

for i, r in df_cpe.iterrows():
    cpeid = r['name']

    if cpeid in connected_cpelist:
        kg_cpe.append([cpeid, "Part", r['part']])
        kg_cpe.append([cpeid, "Vendor", r['vendor']])
        kg_cpe.append([cpeid, "Product", r['product']])
        if (r['target_sw'] != "*"):
            kg_cpe.append([cpeid, "Target_sw", r['target_sw']])
        if (r['target_hw'] != "*"):
            kg_cpe.append([cpeid, "Target_hw", r['target_hw']])

In [17]:
triples = kg_cpe + kg_cve + kg_cwe + kg_cate + kg_view

In [18]:
triples_df = pd.DataFrame(triples, columns=["subject", "predicate", "object"])
triples_df.to_csv(f'./saved/{end_date_folder}/kg_demo_aug2021.csv', index=False)

In [19]:
triples_df = pd.read_csv(f"./saved/{end_date_folder}/kg_demo_aug2021.csv", usecols=["subject", "predicate", "object"])
triples = triples_df[['subject', 'predicate', 'object']].values.tolist()

In [20]:
entities = set()
relations = set()
for triple in triples:
    h, r, t = triple
    entities.add(h)
    entities.add(t)
    relations.add(r)

print(f"Number of entities: {len(entities)}")
print(f"Number of relations: {len(relations)}")
num_entities = len(entities)
num_relations = len(relations)

# must be sorted to produce identical remappings if code block is run again
entity2id = {entity: idx for idx, entity in sorted(enumerate(entities))}
id2entity = {idx: entity for entity, idx in entity2id.items()}

relation2id = {relation: idx for idx, relation in sorted(enumerate(relations))}
id2relation = {idx: relation for relation, idx in relation2id.items()}

triples_id = []
for h, r, t in triples:
    triples_id.append([entity2id[h], relation2id[r], entity2id[t]])
triples_id = np.array(triples_id)


Number of entities: 362687
Number of relations: 18


In [21]:
base_dir = f"saved/{end_date_folder}/id_conversions"

# Create directory tree if it doesn't exist
os.makedirs(base_dir, exist_ok=True)
with open(f"saved/{end_date_folder}/id_conversions/entity2id.pkl", "wb") as f:
    pickle.dump(entity2id, f)

with open(f"saved/{end_date_folder}/id_conversions/id2entity.pkl", "wb") as f:
    pickle.dump(id2entity, f)

with open(f"saved/{end_date_folder}/id_conversions/relation2id.pkl", "wb") as f:
    pickle.dump(relation2id, f)

with open(f"saved/{end_date_folder}/id_conversions/id2relation.pkl", "wb") as f:
    pickle.dump(id2relation, f)

np.save(f"saved/{end_date_folder}/id_conversions/triples_id.npy", triples_id)


In [22]:
with open(f"saved/{end_date_folder}/id_conversions/entity2id.pkl", "rb") as f:
    entity2id = pickle.load(f)

with open(f"saved/{end_date_folder}/id_conversions/id2entity.pkl", "rb") as f:
    id2entity = pickle.load(f)

with open(f"saved/{end_date_folder}/id_conversions/relation2id.pkl", "rb") as f:
    relation2id = pickle.load(f)

with open(f"saved/{end_date_folder}/id_conversions/id2relation.pkl", "rb") as f:
    id2relation = pickle.load(f)

triples_id = np.load(f"saved/{end_date_folder}/id_conversions/triples_id.npy")

## Build the model

In [23]:
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim):
        super(TransE, self).__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.embedding_dim = embedding_dim

        self.entity_embeddings = nn.Embedding(num_entities, embedding_dim)
        self.relation_embeddings = nn.Embedding(num_relations, embedding_dim)
 

        nn.init.xavier_uniform_(self.entity_embeddings.weight.data)
        nn.init.xavier_uniform_(self.relation_embeddings.weight.data)

    def forward(self, pos_triples, neg_triples):
        pos_score = self.score_triples(pos_triples)

        # reshaping b/c >1 neg triple per pos triple 
        # (batch_size, num_neg, 3) -> (batch_size * num_neg, 3)
        batch_size, num_neg, _ = neg_triples.shape
        neg_triples = neg_triples.view(-1, 3)

        neg_score = self.score_triples(neg_triples)
        return pos_score, neg_score

    def predict(self, triples):
        return self.score_triples(triples)

    # transe scoring - higher score = better fit.
    # this can be changed for distmult (h*r*t) or complex
    def score_triples(self, triples):
        h = self.entity_embeddings(triples[:, 0])
        r = self.relation_embeddings(triples[:, 1])
        t = self.entity_embeddings(triples[:, 2])

        # ampligraph defaults to l1 norm
        score = -torch.norm(h + r - t, p=1, dim=1)
        return score

In [24]:
#ensure against integer overflow
def clip_before_exp(x):
    return torch.clamp(x, max=75.0, min=-75.0)


def loss_function(pos_score, neg_score, model, lambda_reg=1e-5):
    # Shape assumptions:
    # pos_score: (batch_size,) or (batch_size, 1)
    # neg_score: (batch_size, num_neg)

    pos_score = pos_score.view(-1, 1)  # (batch_size, 1)
    neg_score = neg_score.view(pos_score.size(0), -1)  # (batch_size, num_neg)

    # Clip before exp like AmpliGraph
    pos_score_clipped = clip_before_exp(pos_score)
    neg_score_clipped = clip_before_exp(neg_score)

    pos_exp = torch.exp(pos_score_clipped)
    neg_exp_sum = torch.sum(torch.exp(neg_score_clipped), dim=1, keepdim=True)

    softmax_score = pos_exp / (pos_exp + neg_exp_sum + 1e-10)
    loss = -torch.log(softmax_score + 1e-10) 
    loss = loss.sum()
    #regularizer
    l3_loss = sum(torch.sum(torch.abs(param) ** 3) for param in [model.entity_embeddings.weight, model.relation_embeddings.weight])
    return loss + lambda_reg * l3_loss

In [25]:
class KGDataset(Dataset):
    def __init__(self, triples):
        self.triples = triples

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        return self.triples[idx]

In [26]:
def generate_batch_negatives(pos_triples, entity_ids, eta=20, corrupt_head_prob=0.5):
    batch_size = pos_triples.shape[0]
    device = pos_triples.device

    # randomly decide whether to corrupt head or tail 1-head, 0-tail
    # batch_size*eta for each neg triple per pos triple in batch
    corrupt_heads = torch.rand(batch_size * eta, device=device) < corrupt_head_prob

    # Repeat positive triples eta times
    pos_repeated = pos_triples.repeat_interleave(eta, dim=0)

    # batch entities to choose from when corrupting
    entity_ids = torch.tensor(entity_ids, device=device)
    # get random entities to use when corrupting h or t
    corrupt_ents = entity_ids[torch.randint(0, len(entity_ids), (batch_size * eta,), device=device)]

    # Clone and corrupt
    neg_triples = pos_repeated.clone()
    neg_triples[corrupt_heads, 0] = corrupt_ents[corrupt_heads]
    neg_triples[~corrupt_heads, 2] = corrupt_ents[~corrupt_heads]

    # reshape back to (batch_size, eta, 3)
    return neg_triples.view(batch_size, eta, 3)


## Monitor the Training

Training a model is often the most time-consuming part of the machine learning process. Large models can take multiple GPUs for days. Expect the training on CPU for this very simple model to take a minute or more.

### Using [ClearML](https://clear.ml/) to Monitor

Please register your account at [https://app.clear.ml/login](https://app.clear.ml/login). After successfully registering your account, get **Application API Credentials**
at: [https://app.clear.ml/settings/workspace-configuration](https://app.clear.ml/settings/workspace-configuration).

Please update `CLEARML_API_ACCESS_KEY` and `CLEARML_API_SECRET_KEY` with your own ClearML **Application API credentials**.

In [27]:
# !pip install clearml
# %env CLEARML_WEB_HOST=https://app.clear.ml/
# %env CLEARML_API_HOST=https://api.clear.ml
# %env CLEARML_FILES_HOST=https://files.clear.ml
# %env CLEARML_API_ACCESS_KEY=<Your_ClearML_API_Key>
# %env CLEARML_API_SECRET_KEY=<Your_ClearML_API_Secret_Key>
# from clearml import Task
# task = Task.init(project_name="fraud detection", task_name="fraud_detection")
!pip install clearml
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
%env CLEARML_API_ACCESS_KEY=197ACIX1GJX033BQ6D42GL7V5NHZ53
%env CLEARML_API_SECRET_KEY=qqfJJDJ3TrjPge2DEzBft6Qwg8dAuQOvY2MOJ5gRIH1UqB-7JGsqfqxMYXG5NY6wraY
from clearml import Task
task = Task.init(project_name="TransE CVE-CWE", task_name="transe_training")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [clearml]m3/4 [clearml]
env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=197ACIX1GJX033BQ6D42GL7V5NHZ53
env: CLEARML_API_SECRET_KEY=qqfJJDJ3TrjPge2DEzBft6Qwg8dAuQOvY2MOJ5gRIH1UqB-7JGsqfqxMYXG5NY6wraY
ClearML Task: created new task id=97f5464e966f4818a50317d2445e6e3e
ClearML results page: https://app.clear.ml/projects/fd5036ee713d4a038ec336544599a5fb/experiments/97f5464e966f4818a50317d2445e6e3e/output/log
ClearML results page: https://app.clear.ml/projects/fd5036ee713d4a038ec336544599a5fb/experiments/97f5464e966f4818a50317d2445e6e3e/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


## Train the model

Training a model is often the most time-consuming part of the machine learning process.  Large models can take multiple GPUs for days.  Expect the training on CPU for this very simple model to take a minute or more.

In [33]:

#mlflow.set_experiment("KG-transe")

In [34]:
#reproducible performance
num = 0
random.seed(num)
np.random.seed(num)
torch.manual_seed(num)
torch.cuda.manual_seed(num)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device:", device)

# # hyperparameters
# embedding_dim = 100
# lambda_reg = 1e-5
# num_epochs = 75
# batches_count = 50
# eta = 20
# learning_rate = 1e-4
# threshold = 0.995

# # dataloader
# batch_size = int(len(triples_id) / batches_count)
# train_dataset = KGDataset(triples_id)
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# # model
# model = TransE(
#     num_entities=len(entity2id),
#     num_relations=len(relation2id),
#     embedding_dim=embedding_dim
# ).to(device)

# optimizer = optim.Adam(model.parameters(), lr=learning_rate)


# # -----------------------------------------------------------------------------
# # Train + MLFLOW
# # -----------------------------------------------------------------------------
# with mlflow.start_run():

#     for epoch in tqdm(range(num_epochs)):
#         model.train()
#         total_loss = 0

#         for pos_triples in train_loader:
#             pos_triples = pos_triples.long().to(device)

#             # unique batch entities for corruption
#             batch_entities = pos_triples[:, [0, 2]].unique().tolist()

#             neg_triples = generate_batch_negatives(
#                 pos_triples, batch_entities, eta=eta
#             ).long().to(device)

#             optimizer.zero_grad()
#             pos_score, neg_score = model(pos_triples, neg_triples)
#             loss = loss_function(pos_score, neg_score, model, lambda_reg)
#             loss.backward()
#             optimizer.step()

#             total_loss += loss.item()

#         avg_loss = total_loss / len(train_loader)
#         print(f"Epoch {epoch+1}/{num_epochs}, Loss={avg_loss:.4f}")
#         mlflow.log_metric("loss", avg_loss, step=epoch)

#     os.makedirs("models/transe/1", exist_ok=True)
#     onnx_path = "models/transe/1/model.onnx"

#     # Dummy example inputs for ONNX export
#     dummy_pos = torch.tensor(
#         [
#             [0, 0, 0],   # any valid triple is fine
#         ],
#         dtype=torch.long
#     )
#     dummy_neg = torch.tensor(
#         [
#             [
#                 [0, 0, 0],
#                 [1, 1, 1],
#             ]
#         ],
#         dtype=torch.long
#     )
#     dummy_pos = dummy_pos.to("cpu")
#     dummy_neg = dummy_neg.to("cpu")
#     model_cpu = model.to("cpu").eval()
#     torch.onnx.export(
#         model_cpu,
#         (dummy_pos, dummy_neg),
#         onnx_path,
#         input_names=["pos_triples", "neg_triples"],
#         output_names=["pos_score", "neg_score"],
#         dynamic_axes={
#             "pos_triples": {0: "batch_size"},
#             "neg_triples": {0: "batch_size", 1: "num_neg"},
#             "pos_score": {0: "batch_size"},
#             "neg_score": {0: "batch_size_times_num_neg"},
#         },
#         opset_version=17,
#     )

#     # log ONNX model to MLflow
#     # mlflow.onnx.log_model(onnx.load(onnx_path), artifact_path="onnx_model")

#     print("ONNX model saved + logged to MLflow:", onnx_path)


In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# hyperparameters
embedding_dim = 100
lambda_reg = 1e-5
num_epochs = 75
batches_count = 50
eta = 20
learning_rate = 1e-4
threshold = 0.995

# dataloader
batch_size = int(len(triples_id) / batches_count)
train_dataset = KGDataset(triples_id)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# model
model = TransE(
    num_entities=len(entity2id),
    num_relations=len(relation2id),
    embedding_dim=embedding_dim
).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------
for epoch in tqdm(range(num_epochs)):
    model.train()
    total_loss = 0

    for pos_triples in train_loader:
        pos_triples = pos_triples.long().to(device)

        # unique batch entities for corruption
        batch_entities = pos_triples[:, [0, 2]].unique().tolist()

        neg_triples = generate_batch_negatives(
            pos_triples, batch_entities, eta=eta
        ).long().to(device)

        optimizer.zero_grad()
        pos_score, neg_score = model(pos_triples, neg_triples)
        loss = loss_function(pos_score, neg_score, model, lambda_reg)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss={avg_loss:.4f}")

# -----------------------------------------------------------------------------
# Export to ONNX
# -----------------------------------------------------------------------------
os.makedirs("models/transe/1", exist_ok=True)
onnx_path = "models/transe/1/model.onnx"

# Dummy example inputs for ONNX export
dummy_pos = torch.tensor(
    [
        [0, 0, 0],
    ],
    dtype=torch.long
)
dummy_neg = torch.tensor(
    [
        [
            [0, 0, 0],
            [1, 1, 1],
        ]
    ],
    dtype=torch.long
)

dummy_pos = dummy_pos.to("cpu")
dummy_neg = dummy_neg.to("cpu")

model_cpu = model.to("cpu").eval()

torch.onnx.export(
    model_cpu,
    (dummy_pos, dummy_neg),
    onnx_path,
    input_names=["pos_triples", "neg_triples"],
    output_names=["pos_score", "neg_score"],
    dynamic_axes={
        "pos_triples": {0: "batch_size"},
        "neg_triples": {0: "batch_size", 1: "num_neg"},
        "pos_score": {0: "batch_size"},
        "neg_score": {0: "batch_size_times_num_neg"},
    },
    opset_version=17,
)

print("ONNX model saved at:", onnx_path)

Using device: cuda


  0%|          | 0/75 [00:00<?, ?it/s]

Epoch 1/75, Loss=43906.2285
Epoch 2/75, Loss=41325.8806
Epoch 3/75, Loss=39371.8705
Epoch 4/75, Loss=37848.7804
Epoch 5/75, Loss=36612.0760
Epoch 6/75, Loss=35564.8825
Epoch 7/75, Loss=34651.3819
Epoch 8/75, Loss=33819.9315
Epoch 9/75, Loss=33061.3915
Epoch 10/75, Loss=32365.6327
Epoch 11/75, Loss=31729.1561
Epoch 12/75, Loss=31109.9522
Epoch 13/75, Loss=30506.7367
Epoch 14/75, Loss=29924.2686
Epoch 15/75, Loss=29375.1047
Epoch 16/75, Loss=28836.6186
Epoch 17/75, Loss=28307.8132
Epoch 18/75, Loss=27803.1022
Epoch 19/75, Loss=27300.0336
Epoch 20/75, Loss=26835.8212
Epoch 21/75, Loss=26377.3130
Epoch 22/75, Loss=25923.7532
Epoch 23/75, Loss=25505.8861
Epoch 24/75, Loss=25071.0866
Epoch 25/75, Loss=24641.3641
Epoch 26/75, Loss=24203.7333
Epoch 27/75, Loss=23812.3921
Epoch 28/75, Loss=23408.8067
Epoch 29/75, Loss=23015.3321
Epoch 30/75, Loss=22644.1988
Epoch 31/75, Loss=22255.8680
Epoch 32/75, Loss=21903.6144
Epoch 33/75, Loss=21543.7463
Epoch 34/75, Loss=21191.3654
Epoch 35/75, Loss=20842

In [37]:
torch.save(model.state_dict(), f"saved/{end_date_folder}/transe_model_nll.pt")
model.eval()

TransE(
  (entity_embeddings): Embedding(362687, 100)
  (relation_embeddings): Embedding(18, 100)
)